# 02 - Layer A: Single-Echelon Stochastic (s, S) via Dynamic Programming

Solves the Bellman equation for each store-item pair. This notebook keeps the 3 debugging iterations from development *visible* rather than only shipping the final working version — the reasoning for why each fix was needed is itself part of the deliverable.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from src.layer_a_dp import (
    solve_infinite_horizon, discretize_normal_demand,
    check_sS_structure, solve_for_row, run_all_pairs,
)
from src.logging_config import get_logger
logger = get_logger(__name__)

## Bellman equation (reference)

$$V(x) = \min_{a \ge 0} \Big\{ K \cdot \mathbb{1}[a>0] + \mathbb{E}_D\big[h\max(x+a-D,0) + p\max(D-x-a,0) + \gamma V(\max(x+a-D,0))\big] \Big\}$$

Solved via infinite-horizon value iteration (not finite-horizon backward induction — see 'Debugging history' below for why that switch mattered).

## Debugging history (do not skip when presenting this project)

1. **Attempt 1** — finite-horizon backward induction, `p` derived from the newsvendor critical ratio using a very small `h`. Result: policy *never* orders — mathematically correct (shortage penalty << fixed order cost $K=50), but useless for demonstrating (s,S).
2. **Attempt 2** — reassigned `p = 30% x unit cost` (lost-margin assumption), dropped the per-unit purchase cost from the objective (standard simplification — purchase cost is paid regardless of *when*, so it doesn't affect the ordering *decision*). Ordering appears, but `S` oscillates wildly across periods — the horizon (T=40-90) is far shorter than the implied reorder cycle (~36 days from the EOQ estimate).
3. **Attempt 3 (final)** — switched to infinite-horizon value iteration. Converges cleanly, 0 structural violations. This is what `solve_infinite_horizon` implements.

## Verify the (s,S) structure emerges (not assumed) on two sample pairs

In [ ]:
model_inputs = pd.read_csv('../data/processed/model_inputs.csv')

case_small = model_inputs.loc[(model_inputs['store']==6) & (model_inputs['item']==5)].iloc[0]
case_medium = model_inputs[model_inputs['demand_mean'].between(45, 55)].iloc[0]

for label, row in [('small demand', case_small), ('medium demand', case_medium)]:
    result = solve_for_row(row)
    print(f"{label}: store={row['store']} item={row['item']} "
          f"s={result['s']} S={result['S']} violations={result['violations']} "
          f"converged={result['converged']} iters={result['iterations']}")

## Run the full batch: all 500 store-item pairs

Progress and any structural violations are logged to `logs/pipeline_<date>.log` via `src/logging_config.py` — check there if this cell runs long.

In [ ]:
policies = run_all_pairs()
policies.describe()

In [ ]:
# Any pairs that failed to converge or violated (s,S) structure - should be empty
problems = policies[(~policies['converged']) | (policies['violations'] > 0)]
print(f'{len(problems)} problematic pairs out of {len(policies)}')
problems